# Conjunto de datos.
El conjunto principal de datos proviene de la Organización Mundial de la Salud [OMS](https://www.who.int/data/gho/data/themes/mortality-and-global-health-estimates/global-health-estimates-leading-causes-of-dalys/) [^1]. Los DALYs (*Disability-adjusted life years*) se reportan a tres escalas diferentes: global, regional y a nivel de país. Para este análisis nos interesa, por el momento, los datos reportados por país desde 2010-2021 de los cuales sólo hay datos para los años 2010, 2015, 2019, 2020 y 2021 (que se debe a que estimados anteriores no se pueden comparar con los actuales). 

Al descargar los archivos ghe2021_daly_bycountry_**** de los años respectivos, fueron movidos a la subcarpeta data_raw, en la subcarpeta de DALYs.

## A. Se cargan las librerías necesarias

In [1]:
import pandas as pd
import numpy as np

## B. Se cargan los datos que se buscan analizar.

In [4]:
# Datos de DALYs

file_paths = {
    2010: 'data_raw/datos_analisis/DALYs/ghe2021_daly_bycountry_2010.xlsx',
    2015: 'data_raw/datos_analisis/DALYs/ghe2021_daly_bycountry_2015.xlsx',
    2019: 'data_raw/datos_analisis/DALYs/ghe2021_daly_bycountry_2019.xlsx',
    2020: 'data_raw/datos_analisis/DALYs/ghe2021_daly_bycountry_2020.xlsx',
    2021: 'data_raw/datos_analisis/DALYs/ghe2021_daly_bycountry_2021.xlsx'
}


# Intervenciones médicas por NTDs
df_interNTD = pd.read_csv('data_raw/datos_analisis/interventions/interventions-ntds-sdgs.csv', encoding='latin-1')

# GDP per capita
df_GDP = pd.read_csv('data_raw/datos_analisis/gdp-per-capita-worldbank/gdp-per-capita.csv', encoding='latin-1')

# Coeficiente Gini
df_gini = pd.read_csv('data_raw/datos_analisis/gini-coefficient-after-tax/gini-coefficient.csv', encoding='latin-1')

# Precipitación anual promedio.
df_preciptacion = pd.read_csv('data_raw/datos_analisis/average-precipitation-per-year/average-precipitation.csv', encoding='latin-1')


## C. Catálogo para enfermedades

In [ ]:
# !!!!!!!!!!!!!!!!! Correr una sola vez para editar manualmente !!!!!!!!!!!!!!!!


# Se carga cualesquiera de las hojas de DALYs
file_cat = file_paths[2000]
edad_cat = '0-4'
cat_enfermedades = pd.read_excel (
    file_cat,
    sheet_name=edad_cat,
    skiprows=range(0, 227)
)

columns_cat = cat_enfermedades.columns[:7]

# Extraer sólo las primeras 7 columnas.
catalogo = cat_enfermedades.iloc[:, :7].copy()  # Las primeras 6 columnas
catalogo.columns = ['sexo', 'codigo_ghe', 'categoria_principal', 
                    'categoria_nivel1', 'categoria_nivel2', 'causa', 'Country or area (See Notes for explanation of colour codes)']

for col in catalogo.columns:
    catalogo[col] = catalogo[col].ffill()

catalogo = catalogo.dropna(subset=['codigo_ghe'])

catalogo.to_csv('data_raw/DALYs_catalogo_enfermedades.csv', index=False)


La edición manual consiste en remover las ambiguedades en la hoja dadas por la estructura de los datos iniciales. La categroría principal 

In [6]:
# Correr esta celda una vez que se realizó el catálogo, recordar no correr dos veces la celda pasada.
catalogo = pd.read_csv('data_raw/DALYs_catalogo_enfermedades.csv', encoding='latin1')
display( catalogo.head() )

,sexo,codigo_ghe,categoria_principal,categoria_nivel1,categoria_nivel2,causa
0,Males,0.0,All Causes,NaN,NaN,NaN
1,Males,10.0,"I.Communicable, maternal, perinatal and nutrit...",NaN,NaN,NaN
2,Males,20.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,NaN,NaN
3,Males,30.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,1.Tuberculosis,Tuberculosis
4,Males,40.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,2.STDs excluding HIV,NaN


In [7]:
len(col_names)

192

## D. Remodelación de Datos

In [8]:
años = [2010, 2015, 2019, 2020, 2021]

col_names = ['sexo', 'codigo_ghe', 'categoria_principal', 'categoria_nivel1', 'categoria_nivel2', 'causa',
             'Country or area (See Notes for explanation of colour codes)', 
             'Afghanistan', 'Albania', 'Algeria', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 
             'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 
             'Belgium', 'Belize', 'Benin', 'Bhutan', 'Bolivia (Plurinational State of)', 'Bosnia and Herzegovina', 
             'Botswana', 'Brazil', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia', 
             'Cameroon', 'Canada', 'Central African Republic', 'Chad', 'Chile', 'China', 'Colombia', 'Comoros', 'Congo', 
             'Costa Rica', "Côte d'Ivoire", 'Croatia', 'Cuba', 'Cyprus', 'Czechia', "Democratic People's Republic of Korea", 
             'Democratic Republic of the Congo', 'Denmark', 'Djibouti', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador', 
             'Equatorial Guinea', 'Eritrea', 'Estonia', 'Eswatini', 'Ethiopia', 'Fiji', 'Finland', 'France', 'Gabon', 'Gambia', 
             'Georgia', 'Germany', 'Ghana', 'Greece', 'Grenada', 'Guatemala', 'Guinea', 'Guinea-Bissau', 'Guyana', 'Haiti', 'Honduras', 
             'Hungary', 'Iceland', 'India', 'Indonesia', 'Iran (Islamic Republic of)', 'Iraq', 'Ireland', 'Israel', 'Italy', 'Jamaica', 'Japan', 
             'Jordan', 'Kazakhstan', 'Kenya', 'Kiribati', 'Kuwait', 'Kyrgyzstan', "Lao People's Democratic Republic", 'Latvia', 'Lebanon', 'Lesotho', 
             'Liberia', 'Libya', 'Lithuania', 'Luxembourg', 'Madagascar', 'Malawi', 'Malaysia', 'Maldives', 'Mali', 'Malta', 'Mauritania', 'Mauritius', 
             'Mexico', 'Micronesia (Federated States of)', 'Mongolia', 'Montenegro', 'Morocco', 'Mozambique', 'Myanmar', 'Namibia', 'Nepal', 'Netherlands', 
             'New Zealand', 'Nicaragua', 'Niger', 'Nigeria', 'North Macedonia', 'Norway', 'occupied Palestinian territory, including east Jerusalem', 'Oman', 
             'Pakistan', 'Panama', 'Papua New Guinea', 'Paraguay', 'Peru', 'Philippines', 'Poland', 'Portugal', 'Puerto Rico', 'Qatar', 'Republic of Korea', 'Republic of Moldova', 
             'Romania', 'Russian Federation', 'Rwanda', 'Saint Lucia', 'Saint Vincent and the Grenadines', 'Samoa', 'Sao Tome and Principe', 'Saudi Arabia', 'Senegal', 'Serbia', 
             'Seychelles', 'Sierra Leone', 'Singapore', 'Slovakia', 'Slovenia', 'Solomon Islands', 'Somalia', 'South Africa', 'South Sudan', 'Spain', 'Sri Lanka', 'Sudan', 'Suriname', 
             'Sweden', 'Switzerland', 'Syrian Arab Republic', 'Tajikistan', 'Thailand', 'Timor-Leste', 'Togo', 'Tonga', 'Trinidad and Tobago', 'Tunisia', 'Türkiye', 'Turkmenistan', 
             'Uganda', 'Ukraine', 'United Arab Emirates', 'United Kingdom', 'United Republic of Tanzania', 'United States of America', 'Uruguay', 'Uzbekistan', 'Vanuatu', 'Venezuela (Bolivarian Republic of)', 
             'Viet Nam', 'Yemen', 'Zambia', 'Zimbabwe']

edades = ['0-4', '5-14','15-29', '30-49', '50-59', '60-69', '70+']

col_order = ['año', 'pais', 'codigo_pais', 'codigo_ghe', 'categoria_principal', 'categoria_nivel1', 'categoria_nivel2', 'causa', 
             'sexo', 'edad', 'DALYs']

# Nos interesan los códigos de las enfermedades correspondientes a Tropicales Desantendidas:
# enfermedad de Chagas, dengue, equinococosis, frambesia o pian (treponematosis),
# fascioliasis, tripanosomiasis africana, leishmaniasis, lepra, filariasis linfática, oncocercosis, rabia,
# esquistosomiasis, geohelmintiasis, cisticercosis, tracoma, sarna y otros ectoparásitos, micetoma y
# micosis profundas.
codigos_ghe = [ 210, 230, 240, 250, 260, 270, 280, 285, 295, 300, 310, 315, 320,340,330, 350, 360, 362, 370 ]

codigos_paises = {
    'Afghanistan': 'AFG', 'Albania': 'ALB', 'Algeria': 'DZA', 'Angola': 'AGO',
    'Antigua and Barbuda': 'ATG', 'Argentina': 'ARG', 'Armenia': 'ARM',
    'Australia': 'AUS', 'Austria': 'AUT', 'Azerbaijan': 'AZE', 'Bahamas': 'BHS',
    'Bahrain': 'BHR', 'Bangladesh': 'BGD', 'Barbados': 'BRB', 'Belarus': 'BLR',
    'Belgium': 'BEL', 'Belize': 'BLZ', 'Benin': 'BEN', 'Bhutan': 'BTN',
    'Bolivia (Plurinational State of)': 'BOL', 'Bosnia and Herzegovina': 'BIH',
    'Botswana': 'BWA', 'Brazil': 'BRA', 'Brunei Darussalam': 'BRN',
    'Bulgaria': 'BGR', 'Burkina Faso': 'BFA', 'Burundi': 'BDI', 'Cabo Verde': 'CPV',
    'Cambodia': 'KHM', 'Cameroon': 'CMR', 'Canada': 'CAN',
    'Central African Republic': 'CAF', 'Chad': 'TCD', 'Chile': 'CHL',
    'China': 'CHN', 'Colombia': 'COL', 'Comoros': 'COM', 'Congo': 'COG',
    'Costa Rica': 'CRI', "Côte d'Ivoire": 'CIV', 'Croatia': 'HRV', 'Cuba': 'CUB',
    'Cyprus': 'CYP', 'Czechia': 'CZE', 'Democratic People\'s Republic of Korea': 'PRK',
    'Democratic Republic of the Congo': 'COD', 'Denmark': 'DNK', 'Djibouti': 'DJI',
    'Dominican Republic': 'DOM', 'Ecuador': 'ECU', 'Egypt': 'EGY',
    'El Salvador': 'SLV', 'Equatorial Guinea': 'GNQ', 'Eritrea': 'ERI',
    'Estonia': 'EST', 'Eswatini': 'SWZ', 'Ethiopia': 'ETH', 'Fiji': 'FJI',
    'Finland': 'FIN', 'France': 'FRA', 'Gabon': 'GAB', 'Gambia': 'GMB',
    'Georgia': 'GEO', 'Germany': 'DEU', 'Ghana': 'GHA', 'Greece': 'GRC',
    'Grenada': 'GRD', 'Guatemala': 'GTM', 'Guinea': 'GIN', 'Guinea-Bissau': 'GNB',
    'Guyana': 'GUY', 'Haiti': 'HTI', 'Honduras': 'HND', 'Hungary': 'HUN',
    'Iceland': 'ISL', 'India': 'IND', 'Indonesia': 'IDN',
    'Iran (Islamic Republic of)': 'IRN', 'Iraq': 'IRQ', 'Ireland': 'IRL',
    'Israel': 'ISR', 'Italy': 'ITA', 'Jamaica': 'JAM', 'Japan': 'JPN',
    'Jordan': 'JOR', 'Kazakhstan': 'KAZ', 'Kenya': 'KEN', 'Kiribati': 'KIR',
    'Kuwait': 'KWT', 'Kyrgyzstan': 'KGZ', 'Lao People\'s Democratic Republic': 'LAO',
    'Latvia': 'LVA', 'Lebanon': 'LBN', 'Lesotho': 'LSO', 'Liberia': 'LBR',
    'Libya': 'LBY', 'Lithuania': 'LTU', 'Luxembourg': 'LUX', 'Madagascar': 'MDG',
    'Malawi': 'MWI', 'Malaysia': 'MYS', 'Maldives': 'MDV', 'Mali': 'MLI',
    'Malta': 'MLT', 'Mauritania': 'MRT', 'Mauritius': 'MUS', 'Mexico': 'MEX',
    'Micronesia (Federated States of)': 'FSM', 'Mongolia': 'MNG', 'Montenegro': 'MNE',
    'Morocco': 'MAR', 'Mozambique': 'MOZ', 'Myanmar': 'MMR', 'Namibia': 'NAM',
    'Nepal': 'NPL', 'Netherlands': 'NLD', 'New Zealand': 'NZL', 'Nicaragua': 'NIC',
    'Niger': 'NER', 'Nigeria': 'NGA', 'North Macedonia': 'MKD', 'Norway': 'NOR',
    'occupied Palestinian territory, including east Jerusalem': 'PSE', 'Oman': 'OMN',
    'Pakistan': 'PAK', 'Panama': 'PAN', 'Papua New Guinea': 'PNG', 'Paraguay': 'PRY',
    'Peru': 'PER', 'Philippines': 'PHL', 'Poland': 'POL', 'Portugal': 'PRT',
    'Puerto Rico': 'PRI', 'Qatar': 'QAT', 'Republic of Korea': 'KOR',
    'Republic of Moldova': 'MDA', 'Romania': 'ROU', 'Russian Federation': 'RUS',
    'Rwanda': 'RWA', 'Saint Lucia': 'LCA', 'Saint Vincent and the Grenadines': 'VCT',
    'Samoa': 'WSM', 'Sao Tome and Principe': 'STP', 'Saudi Arabia': 'SAU',
    'Senegal': 'SEN', 'Serbia': 'SRB', 'Seychelles': 'SYC', 'Sierra Leone': 'SLE',
    'Singapore': 'SGP', 'Slovakia': 'SVK', 'Slovenia': 'SVN', 'Solomon Islands': 'SLB',
    'Somalia': 'SOM', 'South Africa': 'ZAF', 'South Sudan': 'SSD', 'Spain': 'ESP',
    'Sri Lanka': 'LKA', 'Sudan': 'SDN', 'Suriname': 'SUR', 'Sweden': 'SWE',
    'Switzerland': 'CHE', 'Syrian Arab Republic': 'SYR', 'Tajikistan': 'TJK',
    'Thailand': 'THA', 'Timor-Leste': 'TLS', 'Togo': 'TGO', 'Tonga': 'TON',
    'Trinidad and Tobago': 'TTO', 'Tunisia': 'TUN', 'Türkiye': 'TUR',
    'Turkmenistan': 'TKM', 'Uganda': 'UGA', 'Ukraine': 'UKR',
    'United Arab Emirates': 'ARE', 'United Kingdom': 'GBR',
    'United Republic of Tanzania': 'TZA', 'United States of America': 'USA',
    'Uruguay': 'URY', 'Uzbekistan': 'UZB', 'Vanuatu': 'VUT',
    'Venezuela (Bolivarian Republic of)': 'VEN', 'Viet Nam': 'VNM',
    'Yemen': 'YEM', 'Zambia': 'ZMB', 'Zimbabwe': 'ZWE'
}

all_dfs = {}
for año in años:
    file = file_paths[año]
    df_edades = []

    for edad in edades: 
        # Cargar archivos
        print(f"Año: {año}, Edad: {edad}")
        df = pd.read_excel(
            file,
            sheet_name=edad,
            skiprows=range(0, 226),
            # nrows=10 # Quitar para procesar todos los datos
        )
        
        # Se renombran las columnas 
        df.columns = col_names

        # Selección de columnas 
        df = df.drop(columns = ['categoria_principal', 'categoria_nivel1', 'categoria_nivel2', 'causa', 'Country or area (See Notes for explanation of colour codes)'])

        # Transformación a formato tidy
        df = pd.melt(df, id_vars=['codigo_ghe', 'sexo'], var_name='pais', value_name='DALYs')

        # Se agrega el año
        df['año'] = año
        # Se agrega la edad
        df['edad'] = edad
        # Se agrega el código de cada país
        df['codigo_pais'] = df['pais'].map(codigos_paises)

        # Merge con el catálogo de enfermedades de acuerdo con el sexo y el código GHE
        df = pd.merge(df, catalogo, on=['sexo', 'codigo_ghe'], how='inner')

        # Selección de filas de acuerdo con el código de enfermedades.
        df = df[df['codigo_ghe'].isin(codigos_ghe)]

        # Se ordena
        df = df[col_order]
        display(df)

        # Dataframe para edades, dado que se encuentran separadas las hojas
        df_edades.append(df)

        # Merge con datos para obtener variables socioeconómicas y poblacionales. 
    
    # FUERA del bucle de edades, pero DENTRO del bucle de años
    df_anual = pd.concat(df_edades, ignore_index=True)

    # Resultados
    df_anual.to_csv(f"data_clean/DALYs_clean_{año}.csv", index=False)
    all_dfs[año] = df_anual

# Concatenación de todos los Dataframes
df_unificado = pd.concat(all_dfs.values(), ignore_index=True)
df_unificado.to_csv('data_clean/DALYs_clean_unificado.csv', index=False)

display( df_unificado )



Año: 2010, Edad: 0-4


,año,pais,codigo_pais,codigo_ghe,categoria_principal,categoria_nivel1,categoria_nivel2,causa,sexo,edad,DALYs
25,2010,Afghanistan,AFG,210.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,NaN,Males,0-4,1.946223e+01
27,2010,Afghanistan,AFG,230.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,b.Trypanosomiasis,Males,0-4,0.000000e+00
28,2010,Afghanistan,AFG,240.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,c.Chagas disease,Males,0-4,0.000000e+00
29,2010,Afghanistan,AFG,250.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,d.Schistosomiasis,Males,0-4,0.000000e+00
30,2010,Afghanistan,AFG,260.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,e.Leishmaniasis,Males,0-4,3.030321e+00
...,...,...,...,...,...,...,...,...,...,...,...
80485,2010,Zimbabwe,ZWE,340.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,a.Ascariasis,Females,0-4,6.086966e-01
80486,2010,Zimbabwe,ZWE,350.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,b.Trichuriasis,Females,0-4,9.485186e-09
80487,2010,Zimbabwe,ZWE,360.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,c.Hookworm disease,Females,0-4,9.750076e-01
80488,2010,Zimbabwe,ZWE,362.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,d.Food-borne trematodes,Females,0-4,0.000000e+00


Año: 2010, Edad: 5-14


,año,pais,codigo_pais,codigo_ghe,categoria_principal,categoria_nivel1,categoria_nivel2,causa,sexo,edad,DALYs
25,2010,Afghanistan,AFG,210.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,NaN,Males,5-14,2.060825e+01
27,2010,Afghanistan,AFG,230.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,b.Trypanosomiasis,Males,5-14,0.000000e+00
28,2010,Afghanistan,AFG,240.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,c.Chagas disease,Males,5-14,0.000000e+00
29,2010,Afghanistan,AFG,250.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,d.Schistosomiasis,Males,5-14,0.000000e+00
30,2010,Afghanistan,AFG,260.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,e.Leishmaniasis,Males,5-14,6.934876e+00
...,...,...,...,...,...,...,...,...,...,...,...
80485,2010,Zimbabwe,ZWE,340.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,a.Ascariasis,Females,5-14,3.275983e-02
80486,2010,Zimbabwe,ZWE,350.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,b.Trichuriasis,Females,5-14,1.074647e-07
80487,2010,Zimbabwe,ZWE,360.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,c.Hookworm disease,Females,5-14,3.661542e+00
80488,2010,Zimbabwe,ZWE,362.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,d.Food-borne trematodes,Females,5-14,0.000000e+00


Año: 2010, Edad: 15-29


...

Año: 2021, Edad: 60-69


,año,pais,codigo_pais,codigo_ghe,categoria_principal,categoria_nivel1,categoria_nivel2,causa,sexo,edad,DALYs
25,2021,Afghanistan,AFG,210.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,NaN,Males,60-69,2.308833e+00
27,2021,Afghanistan,AFG,230.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,b.Trypanosomiasis,Males,60-69,0.000000e+00
28,2021,Afghanistan,AFG,240.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,c.Chagas disease,Males,60-69,0.000000e+00
29,2021,Afghanistan,AFG,250.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,d.Schistosomiasis,Males,60-69,0.000000e+00
30,2021,Afghanistan,AFG,260.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,e.Leishmaniasis,Males,60-69,1.393067e+00
...,...,...,...,...,...,...,...,...,...,...,...
80485,2021,Zimbabwe,ZWE,340.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,a.Ascariasis,Females,60-69,4.004922e-03
80486,2021,Zimbabwe,ZWE,350.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,b.Trichuriasis,Females,60-69,6.249762e-09
80487,2021,Zimbabwe,ZWE,360.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,c.Hookworm disease,Females,60-69,1.905550e-01
80488,2021,Zimbabwe,ZWE,362.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,d.Food-borne trematodes,Females,60-69,0.000000e+00


Año: 2021, Edad: 70+


,año,pais,codigo_pais,codigo_ghe,categoria_principal,categoria_nivel1,categoria_nivel2,causa,sexo,edad,DALYs
25,2021,Afghanistan,AFG,210.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,NaN,Males,70+,1.558973e+00
27,2021,Afghanistan,AFG,230.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,b.Trypanosomiasis,Males,70+,0.000000e+00
28,2021,Afghanistan,AFG,240.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,c.Chagas disease,Males,70+,0.000000e+00
29,2021,Afghanistan,AFG,250.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,d.Schistosomiasis,Males,70+,0.000000e+00
30,2021,Afghanistan,AFG,260.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,e.Leishmaniasis,Males,70+,9.666499e-01
...,...,...,...,...,...,...,...,...,...,...,...
80485,2021,Zimbabwe,ZWE,340.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,a.Ascariasis,Females,70+,4.103489e-03
80486,2021,Zimbabwe,ZWE,350.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,b.Trichuriasis,Females,70+,2.996110e-09
80487,2021,Zimbabwe,ZWE,360.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,c.Hookworm disease,Females,70+,1.227859e-01
80488,2021,Zimbabwe,ZWE,362.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,d.Food-borne trematodes,Females,70+,0.000000e+00


,año,pais,codigo_pais,codigo_ghe,categoria_principal,categoria_nivel1,categoria_nivel2,causa,sexo,edad,DALYs
0,2010,Afghanistan,AFG,210.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,NaN,Males,0-4,1.946223e+01
1,2010,Afghanistan,AFG,230.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,b.Trypanosomiasis,Males,0-4,0.000000e+00
2,2010,Afghanistan,AFG,240.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,c.Chagas disease,Males,0-4,0.000000e+00
3,2010,Afghanistan,AFG,250.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,d.Schistosomiasis,Males,0-4,0.000000e+00
4,2010,Afghanistan,AFG,260.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,e.Leishmaniasis,Males,0-4,3.030321e+00
...,...,...,...,...,...,...,...,...,...,...,...
246045,2021,Zimbabwe,ZWE,340.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,a.Ascariasis,Females,70+,4.103489e-03
246046,2021,Zimbabwe,ZWE,350.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,b.Trichuriasis,Females,70+,2.996110e-09
246047,2021,Zimbabwe,ZWE,360.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,c.Hookworm disease,Females,70+,1.227859e-01
246048,2021,Zimbabwe,ZWE,362.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,d.Food-borne trematodes,Females,70+,0.000000e+00


## Adición de variables socioecocómicas y climáticas
Las hojas para estas variables se modificaron manualmente para poder realizar un merge de acuerdo con los nombres ya estandarizados para las variables. 

In [25]:
# Definimos el DataFrame unificado.
df_unificado = pd.read_csv('data_clean/DALYs_clean_unificado.csv')

# Definimos variables que nos funcionarán.
col_order = ['año', 'pais', 'codigo_pais', 'codigo_ghe', 'categoria_principal', 'categoria_nivel1', 'categoria_nivel2', 'causa', 
             'sexo', 'edad', 'DALYs', 'intervenciones_ NTDs',	'coeficiente_gini',	'GDP per capita',	'precipitacion_anual']

# Definimos las columnas que nos interesan de cada uno
columnas_gini = ['pais', 'año', 'coeficiente_gini']
columnas_GDP = ['pais', 'codigo_pais', 'año', 'GDP per capita']

# Merge de los conjuntos de datos que nos interesan.

df_unificado = pd.merge(df_unificado, 
                        df_interNTD,
                        on=['pais', 'codigo_pais', 'año'], 
                        how='left'
)

df_unificado = pd.merge(df_unificado, 
                        df_gini[columnas_gini], 
                        on=['pais', 'año'], 
                        how='left'
)


df_unificado = pd.merge(df_unificado, 
                        df_GDP[columnas_GDP], 
                        on=['pais', 'codigo_pais', 'año'], 
                        how='left'
)

df_unificado = pd.merge(df_unificado, 
                        df_preciptacion, 
                        on=['pais', 'codigo_pais', 'año'], 
                        how='left'
)

df_unificado = df_unificado[col_order]

df_unificado.to_csv('data_clean/DALYs_clean_unificado.csv', index=False)
display(df_unificado)


,año,pais,codigo_pais,codigo_ghe,categoria_principal,categoria_nivel1,categoria_nivel2,causa,sexo,edad,DALYs,intervenciones_ NTDs,coeficiente_gini,GDP per capita,precipitacion_anual
0,2010,Afghanistan,AFG,210.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,NaN,Males,0-4,1.946223e+01,12246738.0,0.488102,2848.5862,332.74048
1,2010,Afghanistan,AFG,230.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,b.Trypanosomiasis,Males,0-4,0.000000e+00,12246738.0,0.488102,2848.5862,332.74048
2,2010,Afghanistan,AFG,240.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,c.Chagas disease,Males,0-4,0.000000e+00,12246738.0,0.488102,2848.5862,332.74048
3,2010,Afghanistan,AFG,250.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,d.Schistosomiasis,Males,0-4,0.000000e+00,12246738.0,0.488102,2848.5862,332.74048
4,2010,Afghanistan,AFG,260.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,e.Leishmaniasis,Males,0-4,3.030321e+00,12246738.0,0.488102,2848.5862,332.74048
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
246045,2021,Zimbabwe,ZWE,340.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,a.Ascariasis,Females,70+,4.103489e-03,8147168.0,NaN,4827.0890,788.98170
246046,2021,Zimbabwe,ZWE,350.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,b.Trichuriasis,Females,70+,2.996110e-09,8147168.0,NaN,4827.0890,788.98170
246047,2021,Zimbabwe,ZWE,360.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,c.Hookworm disease,Females,70+,1.227859e-01,8147168.0,NaN,4827.0890,788.98170
246048,2021,Zimbabwe,ZWE,362.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,d.Food-borne trematodes,Females,70+,0.000000e+00,8147168.0,NaN,4827.0890,788.98170


## E. Perfil de Datos
Este perfil se genera en **Google Colab** y se va a generar un perfil para los datos ya remodelados.

In [ ]:
# Sincronizamos Google Drive con Colab. 
# !pip install ydata_profiling # Usar cada que se use Colab
from google.colab import drive
drive.mount('/content/drive')
# También hacemos cargamos ydata_profiling
from ydata_profiling import ProfileReport

ModuleNotFoundError: No module named 'google'

In [ ]:
# Se carga el archivo de datos ya remodelados.
Def_Profile = pd.read_csv('/content/drive/My Drive/DALY_DataAnalysis/data_clean/DALYs_clean_unificado.csv')

# También se cargarán los datos para cada año.
file_paths = {
    2010: '/content/drive/My Drive/DALY_DataAnalysis/data_clean/DALYs_clean_2010.csv',
    2015: '/content/drive/My Drive/DALY_DataAnalysis/data_clean/DALYs_clean_2015.csv',
    2019: '/content/drive/My Drive/DALY_DataAnalysis/data_clean/DALYs_clean_2019.csv',
    2020: '/content/drive/My Drive/DALY_DataAnalysis/data_clean/DALYs_clean_2020.csv',
    2021: '/content/drive/My Drive/DALY_DataAnalysis/data_clean/DALYs_clean_2021.csv'
}
# Definimos una variable de año para automatizar la generación de los perfiles para esos años.
años = [2010, 2015, 2019, 2020, 2021]

for año in años:
    file = file_paths[año]
    print(f"Perfil del año: {año}")
    # Cargar archivos
    df = pd.read_csv(file)
    file_profile="/content/drive/My Drive/DALY_DataAnalysis/profiles/Clean_Profiles/{año}_profiles.html"
    prof = ProfileReport( df, title="Global {año} DALYs Profile", explorative=True)
    prof.to_file(output_file=file_profile)

# Para el perfil del conjunto de datos.
file_profile = "/content/drive/My Drive/DALY_DataAnalysis/profiles/Clean_Profiles/Profile_DALYs.html"
print("Perfil del conjunto de datos")
prof = ProfileReport( Def_Profile, title="Profile_DALYs", explorative=True)
prof.to_file(output_file=file_profile)

,año,pais,codigo_pais,codigo_ghe,categoria_principal,categoria_nivel1,categoria_nivel2,causa,sexo,edad,DALYs
0,2000,Afghanistan,AFG,210.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,NaN,Males,0-4,3.660181e+01
1,2000,Afghanistan,AFG,230.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,b.Trypanosomiasis,Males,0-4,0.000000e+00
2,2000,Afghanistan,AFG,240.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,c.Chagas disease,Males,0-4,0.000000e+00
3,2000,Afghanistan,AFG,250.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,d.Schistosomiasis,Males,0-4,0.000000e+00
4,2000,Afghanistan,AFG,260.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,e.Leishmaniasis,Males,0-4,3.994261e+00
...,...,...,...,...,...,...,...,...,...,...,...
295255,2021,Zimbabwe,ZWE,340.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,a.Ascariasis,Females,70+,4.103489e-03
295256,2021,Zimbabwe,ZWE,350.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,b.Trichuriasis,Females,70+,2.996110e-09
295257,2021,Zimbabwe,ZWE,360.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,c.Hookworm disease,Females,70+,1.227859e-01
295258,2021,Zimbabwe,ZWE,362.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,d.Food-borne trematodes,Females,70+,0.000000e+00


## F. Limpieza de Datos

## Referencias
[^1]: Global Health Estimates 2021: Disease burden by Cause, Age, Sex, by Country and by Region, 2000-2021. Geneva, World Health Organization; 2024.